# Embracing Events

## PSConfEU 2026 - Celebrating 10 Years

![logo](images/logo.png)

### Jeff Hicks | https://jdhitsolutions.github.io

![Microsoft MVP](images/mvp.png)

| Bluesky | Mastodon| GitHub | LinkedIn |
| :-----: | :-----: | :-----: | :-----: |
| [@jdhitsolutions.com](https://bsky.app/profile/jdhitsolutions.com) | [@jeffhicks](https://techhub.social/deck/@JeffHicks) | [jdhitsolutions](https://github.com/jdhitsolutions) | [JefferyHicks](https://www.linkedin.com/in/jefferyhicks/) |

## Sponsor Thank You

![sponsors](images/sponsors.png)

## Events Happen

Windows is an event-driven operating system. (*Some of my material is supported cross-platform*)

- process started
- service stopped
- file created
- a defined event *fires*

Use PowerShell to monitor and respond to events

- Log it
- Respond to it
- Useful for troubleshooting

## Subscriptions

- Create an event subscriber
  - query for some event
  - notify within a given time frame
  - local or remote computer
  - event subscriptions are __temporary__
- Event is added to a local *event queue*
  - event queue is session specific
- Or define a responding action

## Advanced Magic

- forwarded events
- supporting events
- `New-Event` (create your own)

## Limitations and Caveats

- Don't rely on for real-time notifications
- Not intended for permanent monitoring
- I would avoid using for enterprise-level management
- Excellent choice for troubleshooting

## CimIndication Events

- Relies on CIM infrastructure (Windows only)
- Remote event subscriptions are possible
- Use polling
- Monitor `instance` events

In [1]:
Get-CimClass -ClassName __Instance*Event


   NameSpace: ROOT/CIMV2

CimClassName                        CimClassMethods      CimClassProperties
------------                        ---------------      ------------------
__InstanceOperationEvent            {}                   {SECURITY_DESCRIPTOR, TIME_CREATED, TargetInstance}
__InstanceModificationEvent         {}                   {SECURITY_DESCRIPTOR, TIME_CREATED, TargetInstance, PreviousI.
__InstanceCreationEvent             {}                   {SECURITY_DESCRIPTOR, TIME_CREATED, TargetInstance}
__InstanceDeletionEvent             {}                   {SECURITY_DESCRIPTOR, TIME_CREATED, TargetInstance}



In [2]:
$Computername = $env:COMPUTERNAME
$query = "Select * from __InstanceCreationEvent within 10 Where TargetInstance ISA 'win32_process'"

#give your event subscriber a unique identifier to make it easier to manage
$paramHash = @{
  Query            = $query
  ComputerName     = $Computername
  MessageData      = 'A new process was detected'
  SourceIdentifier = 'NewProcess'
}

Register-CimIndicationEvent @paramHash

In [3]:
Get-EventSubscriber


SubscriptionId   : 1
SourceObject     : Microsoft.Management.Infrastructure.CimCmdlets.CimIndicationWatcher
EventName        : CimIndicationArrived
SourceIdentifier : NewProcess
Action           : 
HandlerDelegate  : 
SupportEvent     : False
ForwardEvent     : False



In [4]:
Get-Event


ComputerName     : 
RunspaceId       : ef7d0877-6423-4984-bd6e-f2512a5a741e
EventIdentifier  : 1
Sender           : Microsoft.Management.Infrastructure.CimCmdlets.CimIndicationWatcher
SourceEventArgs  : Microsoft.Management.Infrastructure.CimCmdlets.CimIndicationEventInstanceEventArgs
SourceArgs       : {Microsoft.Management.Infrastructure.CimCmdlets.CimIndicationWatcher, }
SourceIdentifier : NewProcess
TimeGenerated    : 6/2/2026 3:16:02 PM
MessageData      : A new process was detected



In [5]:
Start-Process notepad
Start-Sleep -Seconds 10
Get-Event


ComputerName     : 
RunspaceId       : ef7d0877-6423-4984-bd6e-f2512a5a741e
EventIdentifier  : 1
Sender           : Microsoft.Management.Infrastructure.CimCmdlets.CimIndicationWatcher
SourceEventArgs  : Microsoft.Management.Infrastructure.CimCmdlets.CimIndicationEventInstanceEventArgs
SourceArgs       : {Microsoft.Management.Infrastructure.CimCmdlets.CimIndicationWatcher, }
SourceIdentifier : NewProcess
TimeGenerated    : 6/2/2026 3:16:02 PM
MessageData      : A new process was detected

ComputerName     : 
RunspaceId       : ef7d0877-6423-4984-bd6e-f2512a5a741e
EventIdentifier  : 2
Sender           : Microsoft.Management.Infrastructure.CimCmdlets.CimIndicationWatcher
SourceEventArgs  : Microsoft.Management.Infrastructure.CimCmdlets.CimIndicationEventInstanceEventArgs
SourceArgs       : {Microsoft.Management.Infrastructure.CimCmdlets.CimIndicationWatcher, }
SourceIdentifier : NewProcess
TimeGenerated    : 6/2/2026 3:16:22 PM
MessageData      : A new process was detected

ComputerName 

In [6]:
#give your event subscriber a unique name to make it easier to manage
$e = Get-Event -SourceIdentifier NewProcess | Select -first 1

#details
#time is also part of the event object
$e.TimeGenerated
$e.SourceEventArgs


Tuesday, June 2, 2026 3:16:02 PM

NewEvent  : __InstanceCreationEvent
MachineId : 
Bookmark  : MI_SUBSCRIBE_BOOKMARK_NEWEST
Context   : 




In [7]:
$e.SourceEventArgs.NewEvent | Get-Member -MemberType Properties


   TypeName: Microsoft.Management.Infrastructure.CimInstance#root/cimv2/__InstanceCreationEvent

Name                MemberType Definition
----                ---------- ----------
PSComputerName      Property   string PSComputerName {get;}
SECURITY_DESCRIPTOR Property   byte[] SECURITY_DESCRIPTOR {get;set;}
TargetInstance      Property   CimInstance#Instance TargetInstance {get;set;}
TIME_CREATED        Property   ulong TIME_CREATED {get;set;}



In [8]:
$e.SourceEventArgs.NewEvent.TargetInstance


ProcessId Name                   HandleCount WorkingSetSize VirtualSize
--------- ----                   ----------- -------------- -----------
8168      SearchProtocolHost.exe 434         25952256       2203497140224



In [9]:
foreach ($event in (Get-Event)) {
  $target = $event.SourceEventArgs.NewEvent.TargetInstance
  [PSCustomObject]@{
    Date         = $event.TimeGenerated
    ProcessID    = $target.ProcessID
    ProcessName  = $target.Name
    Path         = $target.Path
    Computername = $target.CSName
  }
}


Date         : 6/2/2026 3:16:02 PM
ProcessID    : 8168
ProcessName  : SearchProtocolHost.exe
Path         : C:\Windows\System32\SearchProtocolHost.exe
Computername : CADENZA

Date         : 6/2/2026 3:16:22 PM
ProcessID    : 11880
ProcessName  : Notepad.exe
Path         : C:\Program 
               Files\WindowsApps\Microsoft.WindowsNotepad_11.2604.5.0_arm64__8wekyb3d8bbwe\Notepad\Notepad.exe
Computername : CADENZA

Date         : 6/2/2026 3:16:22 PM
ProcessID    : 20788
ProcessName  : svchost.exe
Path         : C:\Windows\System32\svchost.exe
Computername : CADENZA



In [10]:
Get-Event | Remove-Event
Unregister-Event NewProcess

In [11]:
# modification events
Stop-Service Bits
$query = "Select * from __InstanceModificationEvent within 5 where TargetInstance ISA 'Win32_Service' AND TargetInstance.Name='BITS' AND TargetInstance.State='Running'"

$paramHash = @{
  Query            = $query
  SourceIdentifier = 'BITSWatch'
  MessageData      = 'BITS has started'
  ComputerName     = $ComputerName
}

Register-CimIndicationEvent @paramHash
Get-EventSubscriber


SubscriptionId   : 2
SourceObject     : Microsoft.Management.Infrastructure.CimCmdlets.CimIndicationWatcher
EventName        : CimIndicationArrived
SourceIdentifier : BITSWatch
Action           : 
HandlerDelegate  : 
SupportEvent     : False
ForwardEvent     : False



In [12]:
Get-Service bits | Start-Service
Start-Sleep -Seconds 6
$e = Get-Event -SourceIdentifier BitsWatch | select -first 1
$e


ComputerName     : 
RunspaceId       : ef7d0877-6423-4984-bd6e-f2512a5a741e
EventIdentifier  : 4
Sender           : Microsoft.Management.Infrastructure.CimCmdlets.CimIndicationWatcher
SourceEventArgs  : Microsoft.Management.Infrastructure.CimCmdlets.CimIndicationEventInstanceEventArgs
SourceArgs       : {Microsoft.Management.Infrastructure.CimCmdlets.CimIndicationWatcher, }
SourceIdentifier : BITSWatch
TimeGenerated    : 6/2/2026 3:17:36 PM
MessageData      : BITS has started



In [13]:
$e.sourceEventArgs.NewEvent | Get-Member


   TypeName: Microsoft.Management.Infrastructure.CimInstance#root/cimv2/__InstanceModificationEvent

Name                      MemberType Definition
----                      ---------- ----------
Dispose                   Method     void Dispose(), void IDisposable.Dispose()
Equals                    Method     bool Equals(System.Object obj)
GetCimSessionComputerName Method     string GetCimSessionComputerName()
GetCimSessionInstanceId   Method     guid GetCimSessionInstanceId()
GetHashCode               Method     int GetHashCode()
GetType                   Method     type GetType()
ToString                  Method     string ToString()
PreviousInstance          Property   CimInstance#Instance PreviousInstance {get;set;}
PSComputerName            Property   string PSComputerName {get;}
SECURITY_DESCRIPTOR       Property   byte[] SECURITY_DESCRIPTOR {get;set;}
TargetInstance            Property   CimInstance#Instance TargetInstance {get;set;}
TIME_CREATED              Property   ulon

In [14]:
diff $e.SourceEventArgs.NewEvent.PreviousInstance $e.SourceEventArgs.NewEvent.TargetInstance -Property State


State   SideIndicator
-----   -------------
Running =>
Stopped <=



In [15]:
#use an action
$query = "Select * from __InstanceModificationEvent within 10 where TargetInstance ISA 'Win32_Service' AND TargetInstance.Name='BITS'"

$action = {
  #can use built-in variables
  # $event = the event object
  # $EventArgs = event.SourceEventArgs

  #case-sensitive
  $i = Get-Date -f 'yyyy-MM-dd_hhmmss_ff'
  $r = [PSCustomObject]@{
    Date         = $event.TimeGenerated
    Previous     = $EventArgs.NewEvent.PreviousInstance | Select-Object State, StartMode, PathName, StartName
    Target       = $EventArgs.NewEvent.TargetInstance | Select-Object State, StartMode, PathName, StartName
    Computername = $EventArgs.NewEvent.PreviousInstance.SystemName
  }
  $r | ConvertTo-Json | Out-File "c:\temp\bits-$i.json"
}

In [16]:
Register-CimIndicationEvent -Query $query -SourceIdentifier 'BITSMonitor2' -Action $action -ComputerName $ComputerName
#Action created as job


Id     Name            PSJobTypeName   State         HasMoreData     Location             Command
--     ----            -------------   -----         -----------     --------             -------
1      BITSMonitor2                    NotStarted    False                                .



In [17]:
#Change the state of the BITS service
(Get-Service bits).state -eq 'Running' ? (Stop-Service Bits -PassThru) : (Start-Service bits -PassThru)


Status   Name               DisplayName
------   ----               -----------
Running  bits               Background Intelligent Transfer Servi.



In [18]:
#No event when using an action
Get-Event -SourceIdentifier BITSMonitor2

Get-Event: 
Line |
   3 |  Get-Event -SourceIdentifier BITSMonitor2
     |  ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
     | Event with source identifier 'BITSMonitor2' does not exist.


In [19]:
dir c:\temp\bits-2026* | Select-Object -last 1 | Get-Content

{
  "Date": "2026-06-02T08:21:01.9017183-04:00",
  "Previous": {
    "State": "Stopped",
    "StartMode": "Manual",
    "PathName": "C:\\Windows\\System32\\svchost.exe -k netsvcs -p",
    "StartName": "LocalSystem"
  },
  "Target": {
    "State": "Running",
    "StartMode": "Auto",
    "PathName": "C:\\Windows\\System32\\svchost.exe -k netsvcs -p",
    "StartName": "LocalSystem"
  },
  "Computername": "CADENZA"
}


In [20]:
Get-EventSubscriber | Unregister-Event
Get-Event | Remove-Event

## High Memory Watcher

```powershell
#requires -version 5.1
#requires -module BurntToast

<#
Watch for processes with high working set values and display a toast message
#>

#polling interval in seconds
$Poll = 10
$query = "Select * from CIM_InstModification within $Poll where TargetInstance ISA 'Win32_Process' AND TargetInstance.WorkingSetSize>=$(900mb)"

$action = {
  #create a log file
  $logPath = 'C:\logs\HighMemLog.txt'
  "[$(Get-Date)] Computername = $($Event.SourceEventArgs.NewEvent.SourceInstance.CSName)" | Out-File -FilePath $logPath -Append
  "[$(Get-Date)] Process = $($Event.SourceEventArgs.NewEvent.SourceInstance.Name)" | Out-File -FilePath $logPath -Append
  "[$(Get-Date)] Command = $($Event.SourceEventArgs.NewEvent.SourceInstance.Commandline)" | Out-File -FilePath $logPath -Append
  "[$(Get-Date)] PID = $($Event.SourceEventArgs.NewEvent.SourceInstance.ProcessID)" | Out-File -FilePath $logPath -Append
  "[$(Get-Date)] WS(MB) = $([math]::Round($Event.SourceEventArgs.NewEvent.SourceInstance.WorkingSetSize/1MB,2))" | Out-File -FilePath $logPath -Append
  "[$(Get-Date)] $('*' * 60)" | Out-File -FilePath $logPath -Append

  #set a toast notification
  $Title = New-BTHeader -title "High Memory Alert"
  $msg = @"
Process = $($Event.SourceEventArgs.NewEvent.SourceInstance.Name)
PID = $($Event.SourceEventArgs.NewEvent.SourceInstance.ProcessID)
WS(MB) = $([math]::Round($Event.SourceEventArgs.NewEvent.SourceInstance.WorkingSetSize/1MB,2))
Date = $($event.TimeGenerated)
"@
  New-BurntToastNotification -Text $msg -AppLogo c:\scripts\mspowershell.png -Header $title
}

Register-CimIndicationEvent -Query $query -SourceIdentifier 'HighProcessMemory' -Action $action
```

In [ ]:
C:\scripts\Watch-ProcessMemory.ps1
Get-EventSubscriber

In [ ]:
Get-Content C:\logs\HighMemLog.txt -tail 6

In [ ]:
Unregister-Event HighProcessMemory

## ProcessTrace

Use process specific classes

In [21]:
Get-CimClass win32_process*trace


   NameSpace: ROOT/CIMV2

CimClassName                        CimClassMethods      CimClassProperties
------------                        ---------------      ------------------
Win32_ProcessTrace                  {}                   {SECURITY_DESCRIPTOR, TIME_CREATED, ParentProcessID, ProcessI.
Win32_ProcessStartTrace             {}                   {SECURITY_DESCRIPTOR, TIME_CREATED, ParentProcessID, ProcessI.
Win32_ProcessStopTrace              {}                   {SECURITY_DESCRIPTOR, TIME_CREATED, ParentProcessID, ProcessI.



In [26]:
#note change in query - not an _instance
$query = "Select * from Win32_ProcessStartTrace WITHIN 10 Where ProcessName = 'pwsh.exe'"
Register-CimIndicationEvent -Query $query -SourceIdentifier ProcessTrace -MessageData "A new pwsh session has started"

In [27]:
#start a new pwsh session
Get-Event -SourceIdentifier ProcessTrace | Tee-Object -Variable e


ComputerName     : 
RunspaceId       : ef7d0877-6423-4984-bd6e-f2512a5a741e
EventIdentifier  : 5
Sender           : Microsoft.Management.Infrastructure.CimCmdlets.CimIndicationWatcher
SourceEventArgs  : Microsoft.Management.Infrastructure.CimCmdlets.CimIndicationEventInstanceEventArgs
SourceArgs       : {Microsoft.Management.Infrastructure.CimCmdlets.CimIndicationWatcher, }
SourceIdentifier : ProcessTrace
TimeGenerated    : 6/2/2026 3:21:09 PM
MessageData      : A new pwsh session has started



In [28]:
#different event result
$e.SourceEventArgs.NewEvent


SECURITY_DESCRIPTOR : 
TIME_CREATED        : 134248800696023757
ParentProcessID     : 26308
ProcessID           : 29620
ProcessName         : pwsh.exe
SessionID           : 3
Sid                 : {1, 5, 0, 0.}
PSComputerName      : 



In [29]:
#get sid
$sid = New-Object System.Security.Principal.SecurityIdentifier($e.SourceEventArgs.NewEvent.Sid, 0)
$sid.Translate([System.Security.Principal.NTAccount]).value

Cadenza\jeff


In [30]:
#clean up
Get-Event | Remove-Event
Get-EventSubscriber | Unregister-Event

## FileSystemWatcher

- Use a .NET class
  - `[System.Io.FileSystemWatcher]`
- watch changes to a folder
- watch changes to a file or files via filtering
- *Should work cross-platform*

In [31]:
New-Object System.IO.FileSystemWatcher


NotifyFilter          : FileName, DirectoryName, LastWrite
Filters               : {}
EnableRaisingEvents   : False
Filter                : *
IncludeSubdirectories : False
InternalBufferSize    : 8192
Path                  : 
Site                  : 
SynchronizingObject   : 
Container             : 



In [32]:
[System.IO.FileSystemWatcher]::new.OverloadDefinitions

System.IO.FileSystemWatcher new()
System.IO.FileSystemWatcher new(string path)
System.IO.FileSystemWatcher new(string path, string filter)


In [33]:
$fsw = [System.IO.FileSystemWatcher]::new("C:\temp")
$fsw


NotifyFilter          : FileName, DirectoryName, LastWrite
Filters               : {}
EnableRaisingEvents   : False
Filter                : *
IncludeSubdirectories : False
InternalBufferSize    : 8192
Path                  : C:\temp
Site                  : 
SynchronizingObject   : 
Container             : 



In [34]:
#need to select an event
$fsw | Get-Member -MemberType Event | Select Name


Name
----
Changed
Created
Deleted
Disposed
Error
Renamed



In [37]:
#enable eventing
$fsw.EnableRaisingEvents = $True
$fsw.IncludeSubdirectories = $True
#register the event - NEW COMMAND
$splat = @{
    InputObject      = $fsw
    SourceIdentifier = 'tempWatch'
    MessageData      = 'A file has changed in C:\temp'
    EventName        = 'Changed'
}
Register-ObjectEvent @splat

In [38]:
1..1000 | Out-File c:\temp\a.txt -Append

In [39]:
#get events
#if a file is created to make the change you might get multiple events
Get-Event -SourceIdentifier tempWatch


ComputerName     : 
RunspaceId       : ef7d0877-6423-4984-bd6e-f2512a5a741e
EventIdentifier  : 7
Sender           : System.IO.FileSystemWatcher
SourceEventArgs  : System.IO.FileSystemEventArgs
SourceArgs       : {System.IO.FileSystemWatcher, a.txt}
SourceIdentifier : tempWatch
TimeGenerated    : 6/2/2026 3:22:33 PM
MessageData      : A file has changed in C:\temp



In [40]:
#limited information
(Get-Event)[0].SourceEventArgs


ChangeType FullPath      Name
---------- --------      ----
   Changed C:\temp\a.txt a.txt



In [41]:
#filter
$fsw.Filter = '*.ps1'
1..1000 | Out-File c:\temp\n.txt -append
code c:\temp\welcome.ps1

In [43]:
Get-Event | Select-Object EventIdentifier,TimeGenerated, @{Name = 'Path'; Expression = { $_.SourceEventArgs.FullPath } }


EventIdentifier TimeGenerated       Path
--------------- -------------       ----
              6 6/2/2026 3:22:33 PM C:\temp\a.txt
              7 6/2/2026 3:22:33 PM C:\temp\a.txt
              9 6/2/2026 3:23:23 PM C:\temp\welcome.ps1
              8 6/2/2026 3:23:23 PM C:\temp\welcome.ps1
             11 6/2/2026 3:23:23 PM C:\temp\welcome.ps1
             10 6/2/2026 3:23:23 PM C:\temp\welcome.ps1



In [46]:
#code for handling multiple related events
$list = [System.Collections.Generic.List[object]]::new()
Get-Event -SourceIdentifier tempWatch | ForEach-Object { $list.Add($_) }
$offSetMS = 500
for ($i = 0 ; $i -lt $list.count; $i++) {
    if ($i+1 -lt $list.count) {
        $test = New-TimeSpan -Start $list[$i].TimeGenerated -End $list[$i + 1].TimeGenerated
        if ($test.TotalMilliseconds -ge $offSetMS) {
            Write-Host "Removing eventID $($list[$i].EventIdentifier)"
            $list.RemoveAt($i)
        }
    }
}
$list | Select-Object EventIdentifier, TimeGenerated, @{Name = 'Path'; Expression = { $_.SourceEventArgs.FullPath } }

Removing eventID 7

EventIdentifier TimeGenerated       Path
--------------- -------------       ----
              9 6/2/2026 3:23:23 PM C:\temp\welcome.ps1
             11 6/2/2026 3:23:23 PM C:\temp\welcome.ps1



In [47]:
Get-EventSubscriber | Unregister-Event
Get-Event | Remove-Event

## PowerShell Engine Events

- Exiting
- OnIdle
- Register-EngineEvent
- Add code to your profile script

```powershell
# Actions when PowerShell quits
# you must quit with Exit
Register-EngineEvent -SourceIdentifier PowerShell.Exiting -Action {
    $Now = Get-Date
        $ps = Get-Process -Id $pid
        [PSCustomObject]@{
            Id           = $ps.Id
            Name         = $ps.Name
            User         = (whoami).ToUpper()
            Computername = [System.Environment]::MachineName
            StartTime    = $ps.StartTime
            EndTime      = $Now
            Runtime      = New-TimeSpan -Start $ps.StartTime -End $Now
        } | Export-Csv -Path c:\logs\ps-exit.csv -Append -NoTypeInformation
 } | Out-Null
 ```

### OnIdle

- use in scripts or modules
- https://github.com/jdhitsolutions/PSClock/blob/main/functions/ConsoleClock.ps1#L68-L114


### Module Close

- register an event to run when the module is removed
- Must use `Remove-Module`

```powershell
#https://github.com/jdhitsolutions/PSBluesky
$OnRemoveScript = {
    #only run this code if the variable is defined
    if ($script:PSCmd) {
        $script:PSCmd.Runspace.Close()
        $script:PSCmd.Runspace.Dispose()
    }
    #clean up type data to avoid errors on re-importing
    'PSBlueskySession', 'PSBlueskySearchResult',
    'PSBlueskyProfile', 'PSBlueskyFollowProfile',
    'PSBlueskyFeedItem', 'PSBlueskyTimelinePost' | Remove-TypeData
    #clean up variables
    Get-Variable -Name bsky*, PDSHost, BSkySession -Exclude BskyPostCache |
    Remove-Variable -ErrorAction SilentlyContinue
}

$ExecutionContext.SessionState.Module.OnRemove += $OnRemoveScript
```

## Daily Management

- for personal or small-scale usage
- Create a script with all of your event subscriptions and related code
- Run as scheduled task or job

```powershell
$action = {
    #Create event subscriptions 
    #keep the job alive!
    Do {
        Start-Sleep -Seconds 1
     } while ($True)
} #close job action

$trigger = New-JobTrigger -AtStartup
$cred = Get-Credential Jeff
$splat = @{
    Name           = 'DailyWatcher'
    ScriptBlock    = $action 
    Trigger        = $trigger 
    RunNow         = $True 
    MaxResultCount = 7 
    Credential     = $cred
}
Register-ScheduledJob @splat
```


## Questions

![melting-head](images/meltinghead.jpg)

## Your Feedback Matters

![QR](images/embracing-events-in-powershell_hicks_1084459_feedback-code.png)